In [1]:
!pip install -f http://h2o-release.s3.amazonaws.com/h2o/latest_stable_Py.html h2o

Looking in links: http://h2o-release.s3.amazonaws.com/h2o/latest_stable_Py.html
   ---------------------------------------- 0.0/266.0 MB ? eta -:--:--
    --------------------------------------- 6.6/266.0 MB 36.0 MB/s eta 0:00:08
   -- ------------------------------------- 14.2/266.0 MB 36.7 MB/s eta 0:00:07
   --- ------------------------------------ 22.0/266.0 MB 36.7 MB/s eta 0:00:07
   ---- ----------------------------------- 29.9/266.0 MB 37.0 MB/s eta 0:00:07
   ----- ---------------------------------- 38.3/266.0 MB 37.4 MB/s eta 0:00:07
   ------ --------------------------------- 46.4/266.0 MB 37.3 MB/s eta 0:00:06
   -------- ------------------------------- 54.5/266.0 MB 37.5 MB/s eta 0:00:06
   --------- ------------------------------ 62.7/266.0 MB 37.5 MB/s eta 0:00:06
   ---------- ----------------------------- 70.5/266.0 MB 37.5 MB/s eta 0:00:06
   ----------- ---------------------------- 78.6/266.0 MB 37.5 MB/s eta 0:00:05
   ------------ --------------------------- 84.4/2

In [1]:
import pandas as pd
import h2o
from h2o.automl import H2OAutoML
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import numpy as np

# -----------------------------
# 1. Load datasets
# -----------------------------
china_df = pd.read_csv("China_dataset.csv")
wqi_df = pd.read_csv("WQI.csv").dropna()

# -----------------------------
# 2. Feature mapping
# -----------------------------
china_features = ['Ammonia (mg/l)', 'Biochemical Oxygen Demand (mg/l)', 'Dissolved Oxygen (mg/l)']
wqi_features = ['NH4+_N', 'COD', 'DO']

# -----------------------------
# 3. Scale input features
# -----------------------------
scaler_china = StandardScaler()
china_X = scaler_china.fit_transform(china_df[china_features])
wqi_X = scaler_china.transform(wqi_df[wqi_features].values)

wqi_y = wqi_df['WQI'].values

# -----------------------------
# 4. H2O setup
# -----------------------------
h2o.init(max_mem_size="4G")  # Khởi tạo H2O cluster

# Chuyển dữ liệu sang H2OFrame
wqi_h2o = h2o.H2OFrame(np.hstack([wqi_X, wqi_y.reshape(-1,1)]))
wqi_h2o.col_names = wqi_features + ['WQI']

# -----------------------------
# 5. Split dữ liệu
# -----------------------------
train, valid = wqi_h2o.split_frame(ratios=[0.8], seed=42)

x = wqi_features
y = 'WQI'

# -----------------------------
# 6. Train model với H2OAutoML
# -----------------------------
aml = H2OAutoML(max_models=10, seed=42, max_runtime_secs=600, nfolds=5)
aml.train(x=x, y=y, training_frame=train, validation_frame=valid)

# -----------------------------
# 7. Evaluate
# -----------------------------
preds = aml.leader.predict(valid)
preds_np = preds.as_data_frame().values.flatten()
true_np = valid[y].as_data_frame().values.flatten()

mse = mean_squared_error(true_np, preds_np)
print("MSE trên tập validation WQI:", mse)

# -----------------------------
# 8. Xem model tốt nhất
# -----------------------------
print(aml.leaderboard)


Checking whether there is an H2O instance running at http://localhost:54321.

C:\Users\Admin\AppData\Local\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


.... not found.
Attempting to start a local H2O server...


H2OStartupError: Cannot find Java. Please install the latest JRE from
http://docs.h2o.ai/h2o/latest-stable/h2o-docs/welcome.html#java-requirements